# 🛡️ Face Anti-Spoofing — 17: Streamlit 앱 실행
> AI Security & Application · 단국대학교 소프트웨어학과  
> 학번: 32214391 · 조현수

---
## ✅ 체크리스트
- [ ] Cell 1: Drive 마운트 + 파일 복사
- [ ] Cell 2: 라이브러리 설치
- [ ] Cell 3: 앱 파일 확인
- [ ] Cell 4: Streamlit 앱 실행

## 📁 필요 파일 (Drive)
```
face-anti-spoofing/
├── src/xai_explainer.py         ← XAI 파이프라인 (threshold=0.65)
├── app/streamlit_app.py         ← Streamlit UI
├── models/stage2_webcam_v2.h5   ← Fine-tuning 완료 모델
└── results/phase4/llava_captions.json
```

## Cell 1 — Drive 마운트 + 파일 복사

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

BASE     = '/content/drive/MyDrive/face-anti-spoofing'
APP_ROOT = '/content/fas_app'   # Colab 로컬 작업 폴더

# 작업 폴더 초기화
if os.path.exists(APP_ROOT):
    shutil.rmtree(APP_ROOT)
os.makedirs(f'{APP_ROOT}/src',    exist_ok=True)
os.makedirs(f'{APP_ROOT}/models', exist_ok=True)
os.makedirs(f'{APP_ROOT}/results/phase4', exist_ok=True)

# ── 필요 파일 복사 ────────────────────────────────────────
files_to_copy = [
    # (Drive 원본 경로, Colab 복사 경로)
    (f'{BASE}/src/xai_explainer.py',
     f'{APP_ROOT}/src/xai_explainer.py'),

    (f'{BASE}/app/streamlit_app.py',
     f'{APP_ROOT}/streamlit_app.py'),

    (f'{BASE}/models/stage2_webcam_v2.h5',
     f'{APP_ROOT}/models/stage2_webcam_v2.h5'),

    (f'{BASE}/results/phase4/llava_captions.json',
     f'{APP_ROOT}/results/phase4/llava_captions.json'),
]

print('=== 파일 복사 ===')
all_ok = True
for src, dst in files_to_copy:
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1024 / 1024
        print(f'  ✅ {os.path.basename(dst):<35} ({size_mb:.1f} MB)')
    else:
        print(f'  ❌ 없음: {src}')
        all_ok = False

print()
if all_ok:
    print('✅ 모든 파일 복사 완료 → Cell 2 진행')
else:
    print('⚠️ 누락 파일 있음 → Drive 경로 확인 필요')

## Cell 2 — 라이브러리 설치

In [ ]:
# Streamlit + ngrok
!pip install -q streamlit pyngrok

# 설치 확인
import streamlit, pyngrok
print(f'streamlit: {streamlit.__version__}')
print(f'pyngrok:   {pyngrok.__version__}')
print('✅ 설치 완료')

## Cell 3 — 앱 파일 & 모델 최종 확인

In [ ]:
import os
import tensorflow as tf

print('=== GPU 확인 ===')
gpus = tf.config.list_physical_devices('GPU')
print(f'  GPU: {gpus[0].name if gpus else "CPU 모드"}')

print('\n=== 앱 파일 확인 ===')
checks = [
    f'{APP_ROOT}/src/xai_explainer.py',
    f'{APP_ROOT}/streamlit_app.py',
    f'{APP_ROOT}/models/stage2_webcam_v2.h5',
    f'{APP_ROOT}/results/phase4/llava_captions.json',
]
for f in checks:
    exists = os.path.exists(f)
    size   = os.path.getsize(f)/1024/1024 if exists else 0
    status = f'✅ ({size:.1f} MB)' if exists else '❌ 없음'
    print(f'  {os.path.basename(f):<40} {status}')

print('\n=== 모델 로드 테스트 ===')
try:
    model = tf.keras.models.load_model(f'{APP_ROOT}/models/stage2_webcam_v2.h5')
    print(f'  ✅ 모델 로드 성공')
    print(f'  입력 shape: {model.input_shape}')
    out_names = [l.name for l in model.layers if l.name in ['binary','spoof']]
    print(f'  출력 레이어: {out_names}')
except Exception as e:
    print(f'  ❌ 모델 로드 실패: {e}')

print('\n=== xai_explainer threshold 확인 ===')
with open(f'{APP_ROOT}/src/xai_explainer.py') as f:
    for i, line in enumerate(f):
        if 'threshold' in line and 'def explain' in line:
            print(f'  {line.strip()}')
        if 'MODEL_PATH' in line and 'models' in line:
            print(f'  {line.strip()}')

## Cell 4 — Streamlit 앱 실행

> **ngrok 토큰 필요:**  
> https://ngrok.com → 가입 → Dashboard → Your Authtoken 복사

In [ ]:
import subprocess, time, sys, os
from pyngrok import ngrok, conf

# ── ngrok 토큰 설정 ───────────────────────────────────────
NGROK_TOKEN = ''   # ← 여기에 ngrok authtoken 붙여넣기

if not NGROK_TOKEN:
    print('⚠️ NGROK_TOKEN을 입력해주세요!')
    print('https://ngrok.com → 가입 → Dashboard → Your Authtoken')
else:
    conf.get_default().auth_token = NGROK_TOKEN

    # ── PYTHONPATH 설정 (xai_explainer import용) ──────────
    env = os.environ.copy()
    env['PYTHONPATH'] = f'{APP_ROOT}/src:' + env.get('PYTHONPATH', '')

    # ── 기존 프로세스 정리 ────────────────────────────────
    os.system('pkill -f streamlit 2>/dev/null; pkill -f ngrok 2>/dev/null')
    time.sleep(1)

    # ── Streamlit 백그라운드 실행 ─────────────────────────
    proc = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run',
         f'{APP_ROOT}/streamlit_app.py',
         '--server.port', '8501',
         '--server.headless', 'true',
         '--server.enableCORS', 'false',
         '--server.enableXsrfProtection', 'false'],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    time.sleep(4)

    # ── ngrok 터널 생성 ───────────────────────────────────
    tunnel = ngrok.connect(8501)
    print('=' * 50)
    print(f'🌐 앱 접속 URL: {tunnel.public_url}')
    print('=' * 50)
    print('위 URL을 브라우저에서 열어주세요!')
    print('\n⏹️ 종료하려면 아래 Cell 5 실행')

## Cell 5 — 앱 종료 (필요 시)

In [ ]:
from pyngrok import ngrok
import os

ngrok.kill()
os.system('pkill -f streamlit')
print('✅ 앱 종료 완료')